In [32]:
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [33]:
# making schema for structured output
class Schema(BaseModel):
    feedback:str = Field(description="short feedback of the essay")
    score:int = Field(description="score essay out of 10",gt=0, lt=11)

In [34]:
# loading model
model = ChatOllama(model="qwen2.5:1.5b")

# making structured model
structured_model = model.with_structured_output(Schema) 


In [35]:
result = model.invoke('give a heading for this - Climate change is one of the biggest challenges facing humanity today. Rising temperatures, extreme weather, and melting ice threaten ecosystems, food security, and human health. Urgent action is needed to reduce carbon emissions, adopt renewable energy, and promote sustainable practices. Collective responsibility can help protect the planet for future generations.')

print(result)

content='"Climate Change: An Urgent Call for Urgent Action and Collective Responsibility"' additional_kwargs={} response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-03-10T21:12:50.3703383Z', 'done': True, 'done_reason': 'stop', 'total_duration': 17038214400, 'load_duration': 3233397600, 'prompt_eval_count': 96, 'prompt_eval_duration': 8945364100, 'eval_count': 17, 'eval_duration': 4642012600, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'} id='lc_run--019cd998-242c-7233-ab9b-d1d7e239dcf9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 96, 'output_tokens': 17, 'total_tokens': 113}


In [36]:
print(result.content)

"Climate Change: An Urgent Call for Urgent Action and Collective Responsibility"


In [37]:
essay = "Climate change is one of the biggest challenges facing humanity today. Rising temperatures, extreme weather, and melting ice threaten ecosystems, food security, and human health. Urgent action is needed to reduce carbon emissions, adopt renewable energy, and promote sustainable practices. Collective responsibility can help protect the planet for future generations."

In [38]:
# creating state

class EssayState(TypedDict):
    essay: str
    cot_feedback: str
    doa_feedback: str
    lang_feedback: str
    scores: Annotated[list[int], operator.add] #using an add reducer
    final_feedback: str
    avg_score: float

In [39]:
# creating node functions

def calc_cot(state: EssayState):
    essay = state['essay']

    prompt = f'give a short feedback and a score to this essay based on clarity of thought - {essay}'

    result = structured_model.invoke(prompt)

    return {
        'cot_feedback': result.feedback,
        'scores': [result.score]
    }

def calc_doa(state: EssayState):
    essay = state['essay']

    prompt = f'give a short feedback and a score to this essay based on depth of analysis - {essay}'

    result = structured_model.invoke(prompt)

    return {
        'doa_feedback': result.feedback,
        'scores': [result.score]
    }

def calc_lang(state: EssayState):
    essay = state['essay']

    prompt = f'give a short feedback and a score to this essay based on language - {essay}'
    result = structured_model.invoke(prompt)

    return {
        'lang_feedback': result.feedback,
        'scores': [result.score]
    }

def gen_final_summary(state: EssayState):
    cot_feedback = state['cot_feedback']
    doa_feedback = state['doa_feedback']
    lang_feedback = state['lang_feedback']

    prompt = f'summarize these feedbacks \n {cot_feedback} \n {doa_feedback} \n {lang_feedback} for the following essay \n {essay}'

    result = model.invoke(prompt).content
    print(result)

    avg_score = sum(state['scores'])/len(state['scores'])

    return {
        'final_feedback': result,
        'avg_score': avg_score
    }


In [40]:
# creating graph and adding nodes and edges

graph = StateGraph(EssayState)

graph.add_node('calc_cot', calc_cot)
graph.add_node('calc_doa', calc_doa)
graph.add_node('calc_lang', calc_lang)
graph.add_node('gen_final_summary', gen_final_summary)

graph.add_edge(START, 'calc_cot')
graph.add_edge(START, 'calc_doa')
graph.add_edge(START, 'calc_lang')
graph.add_edge('calc_cot', 'gen_final_summary')
graph.add_edge('calc_doa', 'gen_final_summary')
graph.add_edge('calc_lang', 'gen_final_summary')
graph.add_edge('gen_final_summary', END)

In [41]:
workflow = graph.compile()

In [42]:
initial_state = {
    'essay' : essay 
}

final_state = workflow.invoke(initial_state)

print(final_state)

### Feedback Summary

#### Essay 1
**Feedback Summary:**
The essay is clear and well-structured, with a clear focus on the urgency of addressing climate change. The main points are well-explained, with credible information providing a strong argument. The solution proposed is relevant and achievable, but the essay could be improved by adding more specific examples and detailed statistics. Overall, it is a good starting point, but it could be more detailed.

**Score: 9/10**

#### Essay 2
**Feedback Summary:**
The essay effectively presents a clear and relevant overview of the impact of climate change on the environment and human life, and the necessity of collective action. The depth of analysis is well-supported with facts and acknowledges future responsibilities. The essay is a strong start and provides a strong overview of the current issues related to climate change. It is clear, concise, and provides straightforward and informative language without any grammatical errors. The struc

In [46]:

print(final_state['scores'])

[4, 9, 5]
